# Bookended graded fit: GPU chunked joint L-BFGS

Warm-start: graded 283.7 eV MLE geometry + instruments (not scale=1 / θ=0).
Opens all free geometry and per-energy scales/thetas under `Transform("YX4")`.
Fit runs on local refloxide with `ReflectModel(..., device="gpu")` and
`minimize_lbfgsb` (exact `nll_and_grad`), chunked 10 iterations at a time with
early stop on a flat nll slope, then an optional f64 CPU polish.

Offline runner: `notebooks/fitting/_bookended_multienergy_polish_gpu.py`
Results: `@models/fitting_results/bookended_multienergy_polish_gpu/`


## 1. Load best fit and data

Same graded pickle and hub XRR load path as `fig_5_graded.ipynb`. The DFT
slab supplies per-energy laboratory corrections only.

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np

from utils import models_root, objectives_retarget_energies, read_fit, read_xrr
from utils.graded_objective import (
    REFLECT_MODEL_INSTRUMENTATION,
    extract_graded_fit_context,
    resolve_data_key,
)
from utils.helpers.fitting_helper import rxr
from utils.helpers.plotting_helper import set_plotting_defaults
from utils.models import configure_refloxide_fitting
from utils.refloxide_view import load_refloxide_fit
from utils.slab_builders import safely_setp_param

configure_refloxide_fitting()
set_plotting_defaults()

ENERGY_ANCHOR = 283.7
E_RES = (283.7, 284.2, 285.1, 287.0, 289.0)
POLS = ("s", "p")
N_Q = 1500
REFLOXIDE_DFT_PATH = models_root / "xrr/znpc/refloxide/refit-refloxide.pkl"

data = read_xrr("reflectivity_data", material="znpc", source="hub")
graded_fit = read_fit("graded/graded_fit.pkl", material="znpc", source="local")
graded_ctx = extract_graded_fit_context(graded_fit)

if not REFLOXIDE_DFT_PATH.exists():
    raise FileNotFoundError(
        f"missing {REFLOXIDE_DFT_PATH} -- same pickle used by fig_5_graded.ipynb"
    )

dft_objective = load_refloxide_fit(REFLOXIDE_DFT_PATH)
dft_corrections = dft_objective.model.corrections

print(f"graded anchor energy: {graded_ctx.energy:g} eV")
print(f"resonant series: {', '.join(f'{e:g}' for e in E_RES)} eV")
print(f"data keys available: {len(data)}")
print(f"DFT slab energies: {list(dft_corrections.energies)}")
print(f"graded varying parameters: {len(graded_fit.varying_parameters())}")

## 2. Retarget graded geometry; seed DFT instrument corrections

Keep the fitted book-ended film and stack from 283.7 eV. Re-evaluate optical
constants at each resonant energy. For `E != 283.7`, overwrite laboratory
scales / theta offsets / background from the DFT-slab channels.

`energy_offset` is taken from the input seed (graded fit at 283.7 eV, DFT slab
elsewhere) and **never varied** — the beamline energy scale is treated as
calibrated. The same holds for `dq`, `q_offset`, and `bkg`.


In [ ]:
def dft_instrument_values(energy: float) -> dict[str, float]:
    """Per-energy DFT-slab corrections plus shared calibrated energy/q terms."""
    channel = dft_corrections.channel_at(float(energy))
    return {
        "scale_s": float(channel.scale_s.value),
        "scale_p": float(channel.scale_p.value),
        "theta_offset_s": float(channel.theta_offset_s.value),
        "theta_offset_p": float(channel.theta_offset_p.value),
        "bkg": float(channel.bkg.value),
        "dq": float(dft_corrections.dq.value or 0.0),
        "q_offset": float(dft_corrections.q_offset.value or 0.0),
        "energy_offset": float(dft_corrections.energy_offset.value or 0.0),
    }


# Calibrated terms: fixed at input for the entire notebook (never opened in CV).
CALIBRATED_INSTRUMENT = ("energy_offset", "dq", "q_offset", "bkg")


def pin_calibrated_instrument(obj, values: dict[str, float] | None = None) -> None:
    """Freeze model + film energy_offset (and other calibrated terms) at input.

    Parameters
    ----------
    obj
        Single-energy anisotropy objective.
    values
        Optional explicit values. When omitted, keep the current parameter values
        and only force ``vary=False``.
    """
    model = obj.model
    for name in CALIBRATED_INSTRUMENT:
        kwargs = {"vary": False}
        if values is not None and name in values:
            kwargs["value"] = values[name]
        safely_setp_param(getattr(model, name), **kwargs)

    film = extract_graded_fit_context(obj).film
    e_off = float(model.energy_offset.value or 0.0) if values is None else float(
        values.get("energy_offset", model.energy_offset.value or 0.0)
    )
    # Film energy_offset tracks the calibrated model offset; never free it.
    if film.energy_offset.constraint is None:
        safely_setp_param(film.energy_offset, value=e_off, vary=False)


def apply_fixed_instrumentation(model, values: dict[str, float]) -> None:
    """Write laboratory corrections onto ``model`` and freeze them."""
    for name in REFLECT_MODEL_INSTRUMENTATION:
        safely_setp_param(getattr(model, name), value=values[name], vary=False)


objectives = objectives_retarget_energies(graded_fit, list(E_RES), data=data)

instrument_table = []
for energy in E_RES:
    obj = objectives[float(energy)]
    if np.isclose(energy, ENERGY_ANCHOR):
        values = {
            name: float(getattr(obj.model, name).value or 0.0)
            for name in REFLECT_MODEL_INSTRUMENTATION
        }
        source = "graded@283.7"
    else:
        values = dft_instrument_values(energy)
        apply_fixed_instrumentation(obj.model, values)
        source = "dft-slab"
    for name in REFLECT_MODEL_INSTRUMENTATION:
        safely_setp_param(getattr(obj.model, name), vary=False)
    pin_calibrated_instrument(obj, values)
    instrument_table.append({"energy": energy, "source": source, **values})

print("instrument seeding (energy_offset fixed / calibrated)")
for row in instrument_table:
    print(
        f"  {row['energy']:6.1f} eV  [{row['source']:11s}]  "
        f"scale_s={row['scale_s']:.4f}  scale_p={row['scale_p']:.4f}  "
        f"ths={row['theta_offset_s']:+.4f}  thp={row['theta_offset_p']:+.4f}  "
        f"E_off={row['energy_offset']:+.4f} (fixed)  bkg={row['bkg']:.3e}"
    )


## 3. Initial comparison on log reflectivity

Data points against the frozen graded prediction at each resonant energy.
One column per polarization; one row per energy.

In [ ]:
def dataset_at(energy: float):
    return data[resolve_data_key(data, energy)]


def q_grid_for(energy: float) -> np.ndarray:
    ds = dataset_at(energy)
    q_max = float(max(np.max(ds.s.x), np.max(ds.p.x)))
    return np.linspace(1e-3, q_max, N_Q)


def plot_resonant_comparison(*, q4: bool = False, figsize=(8.0, 10.5)):
    """Overlay data and graded prediction for every resonant energy.

    Parameters
    ----------
    q4
        When ``True``, plot ``R * q**4`` instead of raw reflectivity.
    figsize
        Matplotlib figure size in inches.
    """
    ylabel = r"$R\,q^{4}$" if q4 else r"$R$"
    title = (
        r"graded@283.7 eV initial condition: $\log(R q^{4})$"
        if q4
        else r"graded@283.7 eV initial condition: $\log R$"
    )
    nrows = len(E_RES)
    fig, axes = plt.subplots(
        nrows,
        2,
        figsize=figsize,
        sharex=True,
        sharey=False,
        constrained_layout=True,
    )
    for i, energy in enumerate(E_RES):
        obj = objectives[float(energy)]
        ds = dataset_at(energy)
        q_model = q_grid_for(energy)
        for j, pol in enumerate(POLS):
            ax = axes[i, j]
            pol_data = getattr(ds, pol)
            q_data = np.asarray(pol_data.x, dtype=float)
            r_data = np.asarray(pol_data.y, dtype=float)
            r_err = np.asarray(pol_data.y_err, dtype=float)
            r_fit = np.asarray(rxr(q_model, obj.model, pol), dtype=float)

            if q4:
                y_data = r_data * q_data**4
                y_err = r_err * q_data**4
                y_fit = r_fit * q_model**4
            else:
                y_data = r_data
                y_err = r_err
                y_fit = r_fit

            ax.errorbar(
                q_data,
                y_data,
                yerr=y_err,
                fmt="o",
                ms=1.5,
                lw=0,
                elinewidth=0.4,
                color="0.35",
                alpha=0.85,
                label="data",
            )
            ax.plot(q_model, y_fit, color="C3", lw=1.1, label="graded init")
            ax.set_yscale("log")
            ax.grid(True, which="both", alpha=0.25)
            if i == 0:
                ax.set_title(f"{pol}-pol")
            if j == 0:
                ax.set_ylabel(f"{energy:g} eV\n{ylabel}")
            if i == nrows - 1:
                ax.set_xlabel(r"$q$ (Å$^{-1}$)")
            if i == 0 and j == 1:
                ax.legend(loc="upper right", fontsize=7, frameon=False)

    fig.suptitle(title, fontsize=11)
    return fig


fig_logR = plot_resonant_comparison(q4=False)
plt.show()

## 4. Same comparison as $\log(R q^{4})$

Identical layout with Fresnel-like falloff removed so Kiessig fringes and
resonance contrast are easier to judge by eye.

In [ ]:
fig_logRq4 = plot_resonant_comparison(q4=True)
plt.show()

## 5. Load multi-energy polish results

Geometry frozen at 283.7 eV; other energies polished with L-BFGS-B on
scale/theta. Re-run the polish script if `status.json` is missing.


In [ ]:
import json
import pickle
from pathlib import Path

import pandas as pd

from utils import models_fitting_results

POLISH_DIR = models_fitting_results / "bookended_multienergy_polish_gpu"
STATUS_PATH = POLISH_DIR / "status.json"
BUNDLE_PATH = POLISH_DIR / "bundle.pkl"
RUN_LOG = POLISH_DIR / "run.log"

if not STATUS_PATH.exists():
    raise FileNotFoundError(
        f"missing {STATUS_PATH}\n"
        "Run: uv run python notebooks/fitting/_bookended_multienergy_polish_gpu.py"
    )

status = json.loads(STATUS_PATH.read_text())
per_energy = pd.read_csv(POLISH_DIR / "per_energy.csv")
geometry = pd.read_csv(POLISH_DIR / "geometry.csv")
instruments = pd.read_csv(POLISH_DIR / "instruments.csv")
summary = pd.read_csv(POLISH_DIR / "summary.csv")

bundle = None
final_map = None
if BUNDLE_PATH.exists():
    with BUNDLE_PATH.open("rb") as fh:
        bundle = pickle.load(fh)
    if status.get("complete"):
        final_map = bundle.get("obj_map")

print(f"polish dir: {POLISH_DIR}")
print(f"complete={status.get('complete')}  protocol={status.get('protocol')}")
if RUN_LOG.exists():
    print(f"run log: {RUN_LOG}")
display(summary.round(4))
print("per-energy RMS before/after")
piv = per_energy.pivot_table(index="energy", columns="phase", values="rms")
display(piv.round(4))
print("frozen geometry")
display(geometry.round(5))


## 6. Instrument corrections before / after polish


In [ ]:
inst_after = instruments.query("phase == 'after'").drop(columns=["phase"])
display(inst_after.round(4))

before = per_energy.query("phase == 'before'").set_index("energy")["rms"]
after = per_energy.query("phase == 'after'").set_index("energy")["rms"]
delta = (after - before).rename("delta_rms")
display(pd.concat([before.rename("rms_before"), after.rename("rms_after"), delta], axis=1).round(4))


## 7. Before / after RMS by energy


In [ ]:
fig_rms, ax = plt.subplots(figsize=(6.0, 3.2), constrained_layout=True)
energies = sorted(per_energy["energy"].unique())
x = np.arange(len(energies))
w = 0.35
bvals = [float(before.loc[e]) for e in energies]
avals = [float(after.loc[e]) for e in energies]
ax.bar(x - w / 2, bvals, w, label="before (DFT/graded seeds)", color="0.6")
ax.bar(x + w / 2, avals, w, label="after L-BFGS polish", color="C3")
ax.set_xticks(x)
ax.set_xticklabels([f"{e:g}" for e in energies])
ax.set_xlabel("energy (eV)")
ax.set_ylabel("YX4 RMS")
ax.legend(frameon=False, fontsize=8)
ax.grid(True, axis="y", alpha=0.3)
ax.set_title("283.7 geometry frozen; other energies instrument-polished")
plt.show()


## 8. Warm-start vs joint L-BFGS in YX4 space ($R q^{4}$)

Same `Transform("YX4")` as the fit. Seed = warm-start (geometry + instruments
before this joint L-BFGS); polish = after opening geometry and all per-energy
scales/thetas with local bounds.


In [ ]:
from contextlib import contextmanager

from refnx.analysis import Transform

Q_MIN = 1e-4
YX4 = Transform("YX4")
FREE_INST = ("scale_s", "scale_p", "theta_offset_s", "theta_offset_p")


@contextmanager
def _instrument_values(model, values: dict[str, float]):
    saved = {name: float(getattr(model, name).value) for name in FREE_INST}
    for name, value in values.items():
        if name in FREE_INST:
            getattr(model, name).value = float(value)
    try:
        yield
    finally:
        for name, value in saved.items():
            getattr(model, name).value = value


def _yx4_curve(q, model, pol: str) -> np.ndarray:
    r = np.asarray(rxr(q, model, pol), dtype=float)
    yt, _ = YX4(q, r)
    return np.asarray(yt, dtype=float)


def plot_yx4_seed_vs_polish(obj_map, inst_before, inst_after, title: str):
    nrows = len(E_RES)
    fig, axes = plt.subplots(
        nrows, 2, figsize=(8.0, 10.5), sharex=True, constrained_layout=True
    )
    for i, energy in enumerate(E_RES):
        obj = obj_map[float(energy)]
        model = obj.model
        ds = obj.data
        q_lo = float(np.min(ds.x))
        q_hi = float(np.max(ds.x))
        q_model = np.linspace(q_lo, q_hi, N_Q)
        before = inst_before[float(energy)]
        after = inst_after[float(energy)]
        for j, pol in enumerate(POLS):
            ax = axes[i, j]
            pol_data = getattr(ds, pol)
            q_data = np.asarray(pol_data.unmasked_data[0], dtype=float)
            r_data = np.asarray(pol_data.unmasked_data[1], dtype=float)
            r_err = np.asarray(pol_data.unmasked_data[2], dtype=float)
            keep = q_data > Q_MIN
            q_data, r_data, r_err = q_data[keep], r_data[keep], r_err[keep]
            y_data, e_data = YX4(q_data, r_data, r_err)

            with _instrument_values(model, before):
                y_seed = _yx4_curve(q_model, model, pol)
            with _instrument_values(model, after):
                y_polish = _yx4_curve(q_model, model, pol)

            ax.errorbar(
                q_data,
                y_data,
                yerr=e_data,
                fmt="o",
                ms=1.5,
                lw=0,
                elinewidth=0.4,
                color="0.35",
                alpha=0.85,
                label="data",
            )
            ax.plot(q_model, y_seed, color="C0", lw=1.0, alpha=0.9, label="seed")
            ax.plot(q_model, y_polish, color="C3", lw=1.1, label="polish")
            ax.set_yscale("log")
            ax.grid(True, which="both", alpha=0.25)
            if i == 0:
                ax.set_title(f"{pol}-pol")
            if j == 0:
                ax.set_ylabel(f"{energy:g} eV\n$R q^{{4}}$")
            if i == nrows - 1:
                ax.set_xlabel(r"$q$ (Å$^{{-1}}$)")
            if i == 0 and j == 1:
                ax.legend(loc="upper right", fontsize=7, frameon=False)
    fig.suptitle(title, fontsize=11)
    return fig


if final_map is None:
    print("obj_map not in bundle yet — re-run the polish script")
else:
    print(f"chi2_all before={bundle['chi2_before']:.3f}  after={bundle['chi2_after']:.3f}")
    print(f"protocol={status.get('protocol')}  transform={status.get('transform', 'YX4')}")
    fig_final = plot_yx4_seed_vs_polish(
        final_map,
        bundle["instrument_before"],
        bundle["instrument_after"],
        r"YX4 ($R q^{4}$): seed instruments vs L-BFGS polish (geom @ 283.7 frozen)",
    )
    plt.show()
